<a href="https://colab.research.google.com/github/zsabro/Batch66/blob/main/HW_task_multiple_agents_and_functions_with_handoffs_and_functions_v003.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai-agents -q #install the opena-AI agents SDK package
!pip install nest_asyncio  # Install nest_asyncio to fix event loop issue

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.9/116.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0/72.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.2 MB/s eta 0:00:00


In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Allow nested event loops in notebook environments

from agents import Agent, Runner, AsyncOpenAI, set_default_openai_client, set_tracing_disabled, set_default_openai_api, function_tool
from google.colab import userdata

# Replace with your actual Gemini API key
gemini_api_key = userdata.get('gemini_api_key')
set_tracing_disabled(True)
set_default_openai_api("chat_completions")

# Set up connection to Gemini AI
external_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
set_default_openai_client(external_client)

# Define Biology Function as a tool
@function_tool
def Biology_Function(prompt: str) -> str:
    """Handles biology-related questions, such as those about cells, organisms, or ecosystems."""
    # Simple logic: return a biology-related answer based on the prompt
    if "cell" in prompt.lower():
        return "Cells are the basic building blocks of life, consisting of a nucleus, cytoplasm, and membrane."
    elif "ecosystem" in prompt.lower():
        return "An ecosystem is a community of living organisms interacting with their environment."
    else:
        return "This is a biology question. Could you provide more details about the topic, like cells or organisms?"

# Define Physics Function as a tool
@function_tool
def Physics_Function(prompt: str) -> str:
    """Handles physics-related questions, such as those about motion, energy, or forces."""
    # Simple logic: return a physics-related answer based on the prompt
    if "motion" in prompt.lower():
        return "Motion is the change in position of an object over time, described by speed and direction."
    elif "energy" in prompt.lower():
        return "Energy is the ability to do work, existing in forms like kinetic and potential energy."
    else:
        return "This is a physics question. Could you provide more details about the topic, like motion or energy?"

# Create Math Agent for math-related questions
math_agent: Agent = Agent(
    name="Math_Agent",
    instructions="You are a math expert. Answer any math-related questions, such as calculations or explaining math concepts, in a clear and simple way.",
    handoff_description="Handles math-related questions, including calculations, algebra, geometry, and math concepts.",
    model="gemini-2.0-flash"
)

# Create History Agent for history-related questions
history_agent: Agent = Agent(
    name="History_Agent",
    instructions="You are a history expert. Answer any history-related questions, such as events or historical figures, in a clear and simple way.",
    handoff_description="Handles history-related questions, including historical events, figures, and timelines.",
    model="gemini-2.0-flash"
)

# Create Science Agent with Biology and Physics tools
science_agent: Agent = Agent(
    name="Science_Agent",
    instructions="You are a science expert. Use the Biology_Function tool for biology-related questions (e.g., cells, organisms) and the Physics_Function tool for physics-related questions (e.g., motion, energy). Answer in a clear and simple way.",
    handoff_description="Handles science-related questions, including biology (cells, organisms) and physics (motion, energy).",
    tools=[Biology_Function, Physics_Function],
    model="gemini-2.0-flash"
)

# Create Triage Agent to delegate to other agents
triage_agent: Agent = Agent(
    name="Triage_Agent",
    instructions="You are a triage assistant. Your job is to read the user's prompt and delegate it to the appropriate agent based on their handoff descriptions. If the prompt is about math, hand it off to Math_Agent. If it’s about history, hand it off to History_Agent. If it’s about science (e.g., biology or physics), hand it off to Science_Agent. If the prompt is irrelevant (not about math, history, or science), respond with: 'Sorry, but I can only answer questions about math, history, or science.'",
    handoffs=[math_agent, history_agent, science_agent],
    model="gemini-2.0-flash"
)

# Function to process the prompt through Triage Agent
def process_prompt(user_prompt):
    # Run the Triage Agent, which will either hand off or respond
    result = Runner.run_sync(triage_agent, user_prompt)
    return result.final_output


In [ ]:
# Test the system with different prompts
test_prompts = [
    "Calculate 6 + 3 * 10 - 20",
    "Who was Muhammad Ali Jinna?",
    "What is a cell?",
    "What is motion?",
    "What's the weather like?"
]

for prompt in test_prompts:
    print(f"Prompt: {prompt}")
    print(f"Response: {process_prompt(prompt)}\n")

## 1. If there is an error "503 - InternalServerError" which is generated when "API" is overloaded or your code overload it by calling different functions

- we have to use "tenacity" library to address this issue.
- The "tenacity library will provide certain modules who will "try" calling the function if failed due to "API overload" issue.

## 2. We will also use "try / except" before running our code to make sure that if there is any run-time error, it does not stop our entire code.

In [ ]:
!pip install openai-agents -q #install openAI Agent SDK package
!pip install nest_asyncio -q # Install nest_asyncio to fix event loop issue
!pip install tenacity -q  # Install tenacity for retry logic

In [ ]:
import nest_asyncio
nest_asyncio.apply()  # Allow nested event loops in notebook environments

from agents import Agent, Runner, AsyncOpenAI, set_default_openai_client, set_tracing_disabled, set_default_openai_api, function_tool
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type
from openai import InternalServerError
from google.colab import userdata

# Replace with your actual Gemini API key
gemini_api_key = userdata.get('gemini_api_key')
set_tracing_disabled(True)
set_default_openai_api("chat_completions")

# Set up connection to Gemini AI
external_client = AsyncOpenAI(
    api_key=gemini_api_key,
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
)
set_default_openai_client(external_client)

# Define Biology Function as a tool
@function_tool
def Biology_Function(prompt: str) -> str:
    """Handles biology-related questions, such as those about cells, organisms, or ecosystems."""
    # Simple logic: return a biology-related answer based on the prompt
    if "cell" in prompt.lower():
        return "Cells are the basic building blocks of life, consisting of a nucleus, cytoplasm, and membrane."
    elif "ecosystem" in prompt.lower():
        return "An ecosystem is a community of living organisms interacting with their environment."
    else:
        return "This is a biology question. Could you provide more details about the topic, like cells or organisms?"

# Define Physics Function as a tool
@function_tool
def Physics_Function(prompt: str) -> str:
    """Handles physics-related questions, such as those about motion, energy, or forces."""
    # Simple logic: return a physics-related answer based on the prompt
    if "motion" in prompt.lower():
        return "Motion is the change in position of an object over time, described by speed and direction."
    elif "energy" in prompt.lower():
        return "Energy is the ability to do work, existing in forms like kinetic and potential energy."
    else:
        return "This is a physics question. Could you provide more details about the topic, like motion or energy?"

# Create Math Agent for math-related questions
math_agent: Agent = Agent(
    name="Math_Agent",
    instructions="You are a math expert. Answer any math-related questions, such as calculations or explaining math concepts, in a clear and simple way.",
    handoff_description="Handles math-related questions, including calculations, algebra, geometry, and math concepts.",
    model="gemini-2.0-flash"
)

# Create History Agent for history-related questions
history_agent: Agent = Agent(
    name="History_Agent",
    instructions="You are a history expert. Answer any history-related questions, such as events or historical figures, in a clear and simple way.",
    handoff_description="Handles history-related questions, including historical events, figures, and timelines.",
    model="gemini-2.0-flash"
)

# Create Science Agent with Biology and Physics tools
science_agent: Agent = Agent(
    name="Science_Agent",
    instructions="You are a science expert. Use the Biology_Function tool for biology-related questions (e.g., cells, organisms) and the Physics_Function tool for physics-related questions (e.g., motion, energy). Answer in a clear and simple way.",
    handoff_description="Handles science-related questions, including biology (cells, organisms) and physics (motion, energy).",
    tools=[Biology_Function, Physics_Function],
    model="gemini-2.0-flash"
)

# Create Triage Agent to delegate to other agents
triage_agent: Agent = Agent(
    name="Triage_Agent",
    instructions="You are a triage assistant. Your job is to read the user's prompt and delegate it to the appropriate agent based on their handoff descriptions. If the prompt is about math, hand it off to Math_Agent. If it’s about history, hand it off to History_Agent. If it’s about science (e.g., biology or physics), hand it off to Science_Agent. If the prompt is irrelevant (not about math, history, or science), respond with: 'Sorry, but I can only answer questions about math, history, or science.'",
    handoffs=[math_agent, history_agent, science_agent],
    model="gemini-2.0-flash"
)

# Function to process the prompt through Triage Agent with retry logic
@retry(
    stop=stop_after_attempt(3),  # Retry up to 3 times
    wait=wait_exponential(multiplier=1, min=1, max=10),  # Wait 1s, 2s, 4s
    retry=retry_if_exception_type(InternalServerError)  # Retry on 503 errors
)
def process_prompt(user_prompt):
    try:
        # Run the Triage Agent, which will either hand off or respond
        result = Runner.run_sync(triage_agent, user_prompt)
        return result.final_output
    except InternalServerError as e:
        # Log the error and re-raise for retry
        print(f"API error: {e}. Retrying...")
        raise


In [ ]:
# Test the system with different prompts
test_prompts = [
    "Calculate 5 + 3",
    "Who was Cleopatra?",
    "What is a cell?",
    "What is motion?",
    "What's the weather like?"
]

for prompt in test_prompts:
    try:
        print(f"Prompt: {prompt}")
        print(f"Response: {process_prompt(prompt)}\n")
    except InternalServerError:
        print(f"Failed to process prompt '{prompt}' after retries due to API overload.\n")

Prompt: Calculate 5 + 3
Response: 5 + 3 = 8


Prompt: Who was Cleopatra?
Response: Cleopatra VII Philopator (usually just called Cleopatra) was the last active ruler of the Ptolemaic Kingdom of Egypt. Basically, she was the last pharaoh of Egypt before it became a province of the Roman Empire.

Here's the breakdown:

*   **She was a Queen:** Cleopatra was a powerful and intelligent ruler who reigned for about three decades (51–30 BC).
*   **She was Egyptian (sort of):** While she ruled Egypt, her family (the Ptolemies) were actually of Greek origin, descendants of one of Alexander the Great's generals.
*   **She was a skilled politician:** She was known for her intelligence, diplomatic skills, and ability to speak multiple languages.
*   **She was involved with powerful Romans:** She's most famous for her relationships with two prominent Roman figures: Julius Caesar and later, Mark Antony. These relationships were crucial for maintaining Egypt's independence and power.
*   **Her story 